In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
import numpy as np

# Check if a GPU is available and set the device accordingly
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [3]:
# Define transformations for the training and testing datasets
# These transformations will convert images to PyTorch tensors and normalize them.
transform = transforms.Compose(
    [
        transforms.ToTensor(), # Convert PIL image to PyTorch tensor
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # Normalize image pixels to range [-1, 1]
    ]
)

# Load the CIFAR-10 training dataset
# download=True ensures the dataset is downloaded to the 'data' directory if not already present.
trainset = torchvision.datasets.CIFAR10(
    root=r'C:\Users\Student\Documents\cifar10_data', train=True, download=True, transform=transform
)

# Create a DataLoader for the training dataset
# This helps in batching and shuffling the data efficiently.
trainloader = DataLoader(trainset, batch_size=4, shuffle=True, num_workers=2)

# Load the CIFAR-10 testing dataset
testset = torchvision.datasets.CIFAR10(
    root=r'C:\Users\Student\Documents\cifar10_data', train=False, download=False, transform=transform
)

# Create a DataLoader for the testing dataset
testloader = DataLoader(testset, batch_size=4, shuffle=False, num_workers=2)

# Define the 10 classes in CIFAR-10
classes = (
    'plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck'
)

print(f"Number of training samples: {len(trainset)}")
print(f"Number of testing samples: {len(testset)}")

100%|██████████| 170M/170M [00:30<00:00, 5.67MB/s]


Number of training samples: 50000
Number of testing samples: 10000


In [4]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # First convolutional layer: 3 input channels (for RGB images), 6 output channels, 5x5 kernel
        self.conv1 = nn.Conv2d(3, 6, 5)
        # Max pooling layer: 2x2 window
        self.pool = nn.MaxPool2d(2, 2)
        # Second convolutional layer: 6 input channels, 16 output channels, 5x5 kernel
        self.conv2 = nn.Conv2d(6, 16, 5)
        # Fully connected layers
        # The input size to the first fully connected layer depends on the output size of the last pooling layer
        # (16 channels * 5x5 feature map size after two conv/pool operations on a 32x32 image)
        self.fc1 = nn.Linear(16 * 5 * 5, 120) # 16 * 5 * 5 is the flattened size of the feature map
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10) # 10 output classes for CIFAR-10

    def forward(self, x):
        # Apply first convolution, then ReLU activation, then pooling
        x = self.pool(nn.functional.relu(self.conv1(x)))
        # Apply second convolution, then ReLU activation, then pooling
        x = self.pool(nn.functional.relu(self.conv2(x)))
        # Flatten the output for the fully connected layers
        x = x.view(-1, 16 * 5 * 5)
        # Apply first fully connected layer, then ReLU activation
        x = nn.functional.relu(self.fc1(x))
        # Apply second fully connected layer, then ReLU activation
        x = nn.functional.relu(self.fc2(x))
        # Apply third fully connected layer (output layer)
        x = self.fc3(x)
        return x

# Instantiate the model and move it to the appropriate device (CPU/GPU)
net = SimpleCNN().to(device)
print(net)

SimpleCNN(
  (conv1): Conv2d(3, 6, kernel_size=(5, 5), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)


In [5]:
# Define the loss function (Cross-Entropy Loss is common for classification)
criterion = nn.CrossEntropyLoss()
# Define the optimizer (SGD with momentum is a good choice)
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

print("Loss function and optimizer initialized.")

Loss function and optimizer initialized.


In [6]:
epochs = 10 # Number of times to iterate over the entire dataset

for epoch in range(epochs):
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # Get the inputs; data is a list of [inputs, labels]
        inputs, labels = data[0].to(device), data[1].to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass: compute predicted outputs by passing inputs to the model
        outputs = net(inputs)
        # Calculate the loss
        loss = criterion(outputs, labels)
        # Backward pass: compute gradient of the loss with respect to model parameters
        loss.backward()
        # Perform a single optimization step (parameter update)
        optimizer.step()

        # Print statistics
        running_loss += loss.item()
        if i % 2000 == 1999:    # Print every 2000 mini-batches
            print(f'Epoch: {epoch + 1}, Batch: {i + 1}, Loss: {running_loss / 2000:.3f}')
            running_loss = 0.0

print('Finished Training')

# Optionally, save the trained model
PATH = r'C:\Users\Student\Documents\cifar_net.pth'
torch.save(net.state_dict(), PATH)
print(f"Model saved to {PATH}")

Epoch: 1, Batch: 2000, Loss: 2.194
Epoch: 1, Batch: 4000, Loss: 1.828
Epoch: 1, Batch: 6000, Loss: 1.623
Epoch: 1, Batch: 8000, Loss: 1.536
Epoch: 1, Batch: 10000, Loss: 1.504
Epoch: 1, Batch: 12000, Loss: 1.449
Epoch: 2, Batch: 2000, Loss: 1.387
Epoch: 2, Batch: 4000, Loss: 1.363
Epoch: 2, Batch: 6000, Loss: 1.338
Epoch: 2, Batch: 8000, Loss: 1.311
Epoch: 2, Batch: 10000, Loss: 1.311
Epoch: 2, Batch: 12000, Loss: 1.273
Epoch: 3, Batch: 2000, Loss: 1.202
Epoch: 3, Batch: 4000, Loss: 1.200
Epoch: 3, Batch: 6000, Loss: 1.192
Epoch: 3, Batch: 8000, Loss: 1.206
Epoch: 3, Batch: 10000, Loss: 1.192
Epoch: 3, Batch: 12000, Loss: 1.165
Epoch: 4, Batch: 2000, Loss: 1.096
Epoch: 4, Batch: 4000, Loss: 1.100
Epoch: 4, Batch: 6000, Loss: 1.106
Epoch: 4, Batch: 8000, Loss: 1.096
Epoch: 4, Batch: 10000, Loss: 1.117
Epoch: 4, Batch: 12000, Loss: 1.116
Epoch: 5, Batch: 2000, Loss: 1.028
Epoch: 5, Batch: 4000, Loss: 1.050
Epoch: 5, Batch: 6000, Loss: 1.037
Epoch: 5, Batch: 8000, Loss: 1.042
Epoch: 5, Ba

RuntimeError: Parent directory C:\Users\Student\Documents does not exist.

In [7]:
# Load the saved model (if you restarted your notebook or want to use a saved model)
# net = SimpleCNN().to(device)
# net.load_state_dict(torch.load(PATH))

correct = 0
total = 0
# Since we're not training, we don't need to calculate the gradients for our outputs
with torch.no_grad():
    for data in testloader:
        images, labels = data[0].to(device), data[1].to(device)
        # Calculate outputs by running images through the network
        outputs = net(images)
        # The class with the highest energy is what we choose as prediction
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the network on the 10000 test images: {100 * correct // total} %')

# Evaluate accuracy for each class
class_correct = list(0. for i in range(10))
class_total = list(0. for i in range(10))
with torch.no_grad():
    for data in testloader:
        images, labels = data[0].to(device), data[1].to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)
        c = (predicted == labels).squeeze()
        for i in range(4):
            label = labels[i]
            class_correct[label] += c[i].item()
            class_total[label] += 1

print('\nAccuracy per class:')
for i in range(10):
    if class_total[i] > 0:
        print(f'Accuracy of {classes[i]:5s} : {100 * class_correct[i] // class_total[i]} %')
    else:
        print(f'No test samples found for {classes[i]:5s}')

Accuracy of the network on the 10000 test images: 62 %

Accuracy per class:
Accuracy of plane : 56.0 %
Accuracy of car   : 72.0 %
Accuracy of bird  : 51.0 %
Accuracy of cat   : 53.0 %
Accuracy of deer  : 53.0 %
Accuracy of dog   : 52.0 %
Accuracy of frog  : 65.0 %
Accuracy of horse : 69.0 %
Accuracy of ship  : 83.0 %
Accuracy of truck : 63.0 %
